In [1]:
from pathlib import Path
import pandas as pd


PROJECT_DIR = Path.cwd().parent


ANALIZ_DIR = PROJECT_DIR / "ciktilar" / "analiz"
OUTPUT_DIR = PROJECT_DIR / "ciktilar"

VERI_DOSYASI = ANALIZ_DIR / "musteri_analiz_tablosu.csv"

musteri_df = pd.read_csv(VERI_DOSYASI)

# Temel veri kontrolü
print("VERİ BOYUTU")
print(musteri_df.shape)

print("\nSÜTUNLAR")
print(musteri_df.columns.tolist())

print("\nVERİ TİPLERİ")
print(musteri_df.dtypes)

print("\nİLK 5 SATIR")
display(musteri_df.head())

print("\nCHURN DAĞILIMI")
print(musteri_df["churn"].value_counts(dropna=False))

print("\nCHURN ORANLARI")
print(musteri_df["churn"].value_counts(normalize=True, dropna=False) * 100)

VERİ BOYUTU
(3974, 18)

SÜTUNLAR
['musteri_id', 'sehir', 'musteri_tipi', 'kayit_kanali', 'son_servis_tarihi', 'frequency', 'monetary', 'recency', 'arac_sayisi', 'ortalama_model_yili', 'aktif_garantili_arac_sayisi', 'geri_bildirim_sayisi', 'metni_olan_geri_bildirim_sayisi', 'farkli_kanal_sayisi', 'ilk_geri_bildirim_tarihi', 'son_geri_bildirim_tarihi', 'hedef_donem_servis_sayisi', 'churn']

VERİ TİPLERİ
musteri_id                           int64
sehir                                  str
musteri_tipi                           str
kayit_kanali                           str
son_servis_tarihi                      str
frequency                            int64
monetary                           float64
recency                              int64
arac_sayisi                          int64
ortalama_model_yili                float64
aktif_garantili_arac_sayisi          int64
geri_bildirim_sayisi                 int64
metni_olan_geri_bildirim_sayisi      int64
farkli_kanal_sayisi                 

,musteri_id,sehir,musteri_tipi,kayit_kanali,son_servis_tarihi,frequency,monetary,recency,arac_sayisi,ortalama_model_yili,aktif_garantili_arac_sayisi,geri_bildirim_sayisi,metni_olan_geri_bildirim_sayisi,farkli_kanal_sayisi,ilk_geri_bildirim_tarihi,son_geri_bildirim_tarihi,hedef_donem_servis_sayisi,churn
0,102288,Ankara,Bireysel,Bayi,2024-12-02,4,9935.32,210,1,2023.0,0,0,0,0,NaN,NaN,2,0
1,103065,Bursa,Bireysel,Bayi,2024-10-10,4,22567.74,263,1,2016.0,0,0,0,0,NaN,NaN,1,0
2,103404,İstanbul,Bireysel,Bayi,2025-06-20,4,47712.88,10,1,2024.0,1,1,1,1,2023-09-23,2023-09-23,3,0
3,102505,İstanbul,Bireysel,Bayi,2025-01-13,7,73175.03,168,1,2015.0,0,0,0,0,NaN,NaN,2,0
4,102599,Kocaeli,Bireysel,Bayi,2025-03-18,7,85006.86,104,1,2018.0,0,0,0,0,NaN,NaN,3,0



CHURN DAĞILIMI
churn
0    3316
1     658
Name: count, dtype: int64

CHURN ORANLARI
churn
0    83.442375
1    16.557625
Name: proportion, dtype: float64


In [2]:
# 1. Modelde kullanılabilecek aday özellikler
FEATURES = [
    "sehir",
    "musteri_tipi",
    "kayit_kanali",
    "frequency",
    "monetary",
    "recency",
    "arac_sayisi",
    "ortalama_model_yili",
    "aktif_garantili_arac_sayisi",
    "geri_bildirim_sayisi",
    "metni_olan_geri_bildirim_sayisi",
    "farkli_kanal_sayisi"
]

TARGET = "churn"

# 2. Kesinlikle modele girmemesi gereken sütunlar
EXCLUDED_COLUMNS = [
    "musteri_id",
    "son_servis_tarihi",
    "ilk_geri_bildirim_tarihi",
    "son_geri_bildirim_tarihi",
    "hedef_donem_servis_sayisi"
]

# 3. Sütunların varlığı kontrolü
assert all(col in musteri_df.columns for col in FEATURES)
assert TARGET in musteri_df.columns
assert all(col in musteri_df.columns for col in EXCLUDED_COLUMNS)

print("MODEL ÖZELLİKLERİ")
print(FEATURES)

print("\nMODEL DIŞINDA TUTULAN SÜTUNLAR")
print(EXCLUDED_COLUMNS)

# 4. Özelliklerde eksik değer kontrolü
print("\nÖZELLİKLERDE EKSİK DEĞERLER")
display(
    musteri_df[FEATURES]
    .isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame("eksik_deger_sayisi")
)

# 5. Sayısal özelliklerin temel istatistikleri
print("\nSAYISAL ÖZELLİKLERİN İSTATİSTİKLERİ")
display(musteri_df[FEATURES].select_dtypes(include="number").describe().T)

# 6. Tarih ve recency tutarlılığı
musteri_df["son_servis_tarihi"] = pd.to_datetime(
    musteri_df["son_servis_tarihi"],
    errors="coerce"
)

KESIM_TARIHI = pd.Timestamp("2025-06-30")

hesaplanan_recency = (
    KESIM_TARIHI - musteri_df["son_servis_tarihi"]
).dt.days

recency_fark = (
    musteri_df["recency"] - hesaplanan_recency
).abs()

print("\nZAMAN TUTARLILIĞI")
print("Geçersiz son servis tarihi:", musteri_df["son_servis_tarihi"].isna().sum())
print("Recency ile tarih hesabı uyuşmayan müşteri:", (recency_fark > 0).sum())
print("Maksimum recency farkı:", recency_fark.max(), "gün")

# 7. Hedef dönemine ait değişkenin özelliklerde bulunmaadığı doğrulanması
assert "hedef_donem_servis_sayisi" not in FEATURES
assert TARGET not in FEATURES
assert "musteri_id" not in FEATURES

print("\nSızıntı kontrolü: Temel sütun kontrolleri başarılı.")

MODEL ÖZELLİKLERİ
['sehir', 'musteri_tipi', 'kayit_kanali', 'frequency', 'monetary', 'recency', 'arac_sayisi', 'ortalama_model_yili', 'aktif_garantili_arac_sayisi', 'geri_bildirim_sayisi', 'metni_olan_geri_bildirim_sayisi', 'farkli_kanal_sayisi']

MODEL DIŞINDA TUTULAN SÜTUNLAR
['musteri_id', 'son_servis_tarihi', 'ilk_geri_bildirim_tarihi', 'son_geri_bildirim_tarihi', 'hedef_donem_servis_sayisi']

ÖZELLİKLERDE EKSİK DEĞERLER


,eksik_deger_sayisi
sehir,0
musteri_tipi,0
kayit_kanali,0
frequency,0
monetary,0
recency,0
arac_sayisi,0
ortalama_model_yili,0
aktif_garantili_arac_sayisi,0
geri_bildirim_sayisi,0



SAYISAL ÖZELLİKLERİN İSTATİSTİKLERİ


,count,mean,std,min,25%,50%,75%,max
frequency,3974.0,6.306241,3.193102,1.00,5.00,6.000,7.00,2.400000e+01
monetary,3974.0,82701.024884,109220.951525,-15893.88,46887.26,67419.975,96226.64,2.941494e+06
recency,3974.0,116.831656,84.834237,0.00,49.00,102.000,168.00,4.130000e+02
arac_sayisi,3974.0,1.139406,0.346413,1.00,1.00,1.000,1.00,2.000000e+00
ortalama_model_yili,3974.0,2018.632612,3.380520,2013.00,2016.00,2019.000,2021.50,2.025000e+03
aktif_garantili_arac_sayisi,3974.0,0.263211,0.455600,0.00,0.00,0.000,1.00,2.000000e+00
geri_bildirim_sayisi,3974.0,0.154504,0.399198,0.00,0.00,0.000,0.00,4.000000e+00
metni_olan_geri_bildirim_sayisi,3974.0,0.151736,0.395514,0.00,0.00,0.000,0.00,4.000000e+00
farkli_kanal_sayisi,3974.0,0.149723,0.382052,0.00,0.00,0.000,0.00,4.000000e+00



ZAMAN TUTARLILIĞI
Geçersiz son servis tarihi: 0
Recency ile tarih hesabı uyuşmayan müşteri: 0
Maksimum recency farkı: 0 gün

Sızıntı kontrolü: Temel sütun kontrolleri başarılı.


Dikkat edilmesi gereken iki konu
1. monetary değişkeninde negatif değerler var. Önceki veri incelemede bunların servis tutarlarında düzeltme veya iade kaynaklı olabileceği değerlendirildi. Şimdilik silmey veya sıfırlama yapılmayacak. Model sonuçlarında etkileri incelenecek.
2. monetary maksimum değeri 2.941.494 TL. Ortalama ile medyan arasında belirgin fark var.

In [3]:
from sklearn.model_selection import train_test_split


X = musteri_df[FEATURES].copy()
y = musteri_df[TARGET].copy()

# Eğitim ve test kümelerine ayır
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("EĞİTİM VE TEST VERİSİ")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

# Churn oranları
dagilim = pd.DataFrame({
    "Eğitim": y_train.value_counts(normalize=True) * 100,
    "Test": y_test.value_counts(normalize=True) * 100
}).sort_index()

dagilim.index.name = "churn"

print("\nCHURN ORANLARI (%)")
display(dagilim)

# Müşteri sayılarının kontrolü
print("\nTOPLAM MÜŞTERİ:", len(X_train) + len(X_test))
print("EĞİTİM + TEST AYRIMI:", len(X_train), "+", len(X_test))

assert len(X_train) + len(X_test) == len(musteri_df)
assert set(X_train.index).isdisjoint(set(X_test.index))
assert y_train.value_counts().index.equals(y_test.value_counts().index)

print("\nKontroller başarılı: Eğitim ve test kümeleri birbirinden ayrıdır.")

EĞİTİM VE TEST VERİSİ
X_train: (3179, 12)
X_test : (795, 12)
y_train: (3179,)
y_test : (795,)

CHURN ORANLARI (%)


,Eğitim,Test
churn,,
0,83.453916,83.396226
1,16.546084,16.603774



TOPLAM MÜŞTERİ: 3974
EĞİTİM + TEST AYRIMI: 3179 + 795

Kontroller başarılı: Eğitim ve test kümeleri birbirinden ayrıdır.


## Baseline Mdel

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Baseline model: Her müşteriyi churn olmayan olarak tahmin et
baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train, y_train)


y_pred_baseline = baseline_model.predict(X_test)


baseline_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_baseline),
    "Precision": precision_score(y_test, y_pred_baseline, zero_division=0),
    "Recall": recall_score(y_test, y_pred_baseline, zero_division=0),
    "F1-score": f1_score(y_test, y_pred_baseline, zero_division=0)
}

print("BASELINE MODEL PERFORMANSI")
for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nKARIŞIKLIK MATRİSİ")
print(confusion_matrix(y_test, y_pred_baseline))

print("\nSINIFLANDIRMA RAPORU")
print(classification_report(
    y_test,
    y_pred_baseline,
    zero_division=0
))

BASELINE MODEL PERFORMANSI
Accuracy: 0.8340
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000

KARIŞIKLIK MATRİSİ
[[663   0]
 [132   0]]

SINIFLANDIRMA RAPORU
              precision    recall  f1-score   support

           0       0.83      1.00      0.91       663
           1       0.00      0.00      0.00       132

    accuracy                           0.83       795
   macro avg       0.42      0.50      0.45       795
weighted avg       0.70      0.83      0.76       795



## İlk model: Random Forest

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


CATEGORICAL_FEATURES = [
    "sehir",
    "musteri_tipi",
    "kayit_kanali"
]

NUMERICAL_FEATURES = [
    col for col in FEATURES
    if col not in CATEGORICAL_FEATURES
]


preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"),
         CATEGORICAL_FEATURES),
        ("numerical", "passthrough", NUMERICAL_FEATURES)
    ]
)


rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])


rf_model.fit(X_train, y_train)

# Test tahminleri
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# Performans metrikleri
rf_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "Precision": precision_score(y_test, y_pred_rf, zero_division=0),
    "Recall": recall_score(y_test, y_pred_rf, zero_division=0),
    "F1-score": f1_score(y_test, y_pred_rf, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
    "PR-AUC": average_precision_score(y_test, y_prob_rf)
}

print("RANDOM FOREST MODEL PERFORMANSI")
for metric, value in rf_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nKARIŞIKLIK MATRİSİ")
print(confusion_matrix(y_test, y_pred_rf))

print("\nSINIFLANDIRMA RAPORU")
print(classification_report(
    y_test,
    y_pred_rf,
    zero_division=0
))

RANDOM FOREST MODEL PERFORMANSI
Accuracy: 0.7560
Precision: 0.3418
Recall: 0.5076
F1-score: 0.4085
ROC-AUC: 0.7144
PR-AUC: 0.4651

KARIŞIKLIK MATRİSİ
[[534 129]
 [ 65  67]]

SINIFLANDIRMA RAPORU
              precision    recall  f1-score   support

           0       0.89      0.81      0.85       663
           1       0.34      0.51      0.41       132

    accuracy                           0.76       795
   macro avg       0.62      0.66      0.63       795
weighted avg       0.80      0.76      0.77       795



In [6]:
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# 5 katlı çapraz doğrulama
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Eğitim verisi için out-of-fold churn olasılıkları
y_prob_oof = cross_val_predict(
    rf_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Farklı karar eşiklerini k
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

threshold_results = []

for threshold in thresholds:
    y_pred = (y_prob_oof >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_train, y_pred),
        "precision": precision_score(y_train, y_pred, zero_division=0),
        "recall": recall_score(y_train, y_pred, zero_division=0),
        "f1": f1_score(y_train, y_pred, zero_division=0),
        "tahmin_edilen_churn": int(y_pred.sum())
    })

threshold_df = pd.DataFrame(threshold_results)

print("EĞİTİM VERİSİ - 5 KATLI ÇAPRAZ DOĞRULAMA")
display(threshold_df.round(4))

EĞİTİM VERİSİ - 5 KATLI ÇAPRAZ DOĞRULAMA


,threshold,accuracy,precision,recall,f1,tahmin_edilen_churn
0,0.20,0.2831,0.1823,0.9563,0.3062,2759
1,0.25,0.3756,0.1982,0.9106,0.3255,2417
2,0.30,0.4674,0.2152,0.8384,0.3425,2049
3,0.35,0.5703,0.2461,0.7738,0.3734,1654
4,0.40,0.6640,0.2915,0.7205,0.4151,1300
5,0.45,0.7310,0.3404,0.6673,0.4509,1031
6,0.50,0.7707,0.3758,0.5837,0.4572,817
7,0.55,0.8047,0.4230,0.4962,0.4567,617
8,0.60,0.8304,0.4855,0.4125,0.4460,447


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Eğitim verisiyle eğitilmiş Random Forest'ın test olasılıkları
y_prob_test = rf_model.predict_proba(X_test)[:, 1]

# CV sonuçlarına göre belirlediğimiz aday eşikler
TEST_THRESHOLDS = [0.50, 0.55]

test_results = []

for threshold in TEST_THRESHOLDS:

    y_pred = (y_prob_test >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    test_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "predicted_churn": int(y_pred.sum())
    })

test_results_df = pd.DataFrame(test_results)

print("TEST KÜMESİ - EŞİK KARŞILAŞTIRMASI")
display(test_results_df.round(4))

print("\nTEST KÜMESİ - ROC-AUC:", round(roc_auc_score(y_test, y_prob_test), 4))
print("TEST KÜMESİ - PR-AUC:", round(average_precision_score(y_test, y_prob_test), 4))

TEST KÜMESİ - EŞİK KARŞILAŞTIRMASI


,threshold,accuracy,precision,recall,f1,true_negative,false_positive,false_negative,true_positive,predicted_churn
0,0.50,0.7560,0.3418,0.5076,0.4085,534,129,65,67,196
1,0.55,0.7862,0.3812,0.4621,0.4178,564,99,71,61,160



TEST KÜMESİ - ROC-AUC: 0.7144
TEST KÜMESİ - PR-AUC: 0.4651


In [8]:
# Random Forest değişken önemleri

rf_classifier = rf_model.named_steps["classifier"]
fitted_preprocessor = rf_model.named_steps["preprocessor"]

feature_names_transformed = fitted_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "donusturulmus_degisken": feature_names_transformed,
    "importance": rf_classifier.feature_importances_
})


def original_feature_name(feature):
    if feature.startswith("categorical__"):
        encoded_name = feature.replace("categorical__", "", 1)

        for col in sorted(CATEGORICAL_FEATURES, key=len, reverse=True):
            if encoded_name.startswith(col + "_"):
                return col

        raise ValueError(f"Değişken eşleştirilemedi: {feature}")

    if feature.startswith("numerical__"):
        return feature.replace("numerical__", "", 1)

    return feature


importance_df["orijinal_degisken"] = (
    importance_df["donusturulmus_degisken"]
    .apply(original_feature_name)
)

grouped_importance = (
    importance_df
    .groupby("orijinal_degisken")["importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

grouped_importance.columns = ["degisken", "importance"]
grouped_importance["importance_yuzde"] = (
    grouped_importance["importance"] * 100
)

print("TÜM DEĞİŞKENLERİN ÖNEM SIRALAMASI")
display(grouped_importance.round(4))

print("\nEN ÖNEMLİ 5 DEĞİŞKEN")
display(grouped_importance.head(5).round(4))

TÜM DEĞİŞKENLERİN ÖNEM SIRALAMASI


,degisken,importance,importance_yuzde
0,recency,0.4013,40.1312
1,monetary,0.1496,14.9629
2,ortalama_model_yili,0.1315,13.1465
3,frequency,0.1030,10.2994
4,sehir,0.0709,7.0861
5,kayit_kanali,0.0440,4.4022
6,musteri_tipi,0.0423,4.2281
7,aktif_garantili_arac_sayisi,0.0273,2.7316
8,arac_sayisi,0.0112,1.1158
9,geri_bildirim_sayisi,0.0066,0.6611



EN ÖNEMLİ 5 DEĞİŞKEN


,degisken,importance,importance_yuzde
0,recency,0.4013,40.1312
1,monetary,0.1496,14.9629
2,ortalama_model_yili,0.1315,13.1465
3,frequency,0.1030,10.2994
4,sehir,0.0709,7.0861


| Sıra | Değişken | Önem | İş Açısından Yorum |
| --- | --- | --- | --- |
| **1** | `recency` | %40,13 | Müşterinin son servisinden bu yana geçen süre. Uzun süredir servise gelmeyen müşteriler daha yüksek terk riski taşıyabilir. |
| **2** | `monetary` | %14,96 | Müşterinin geçmiş servis harcaması. Müşterinin servis ağıyla geçmiş ekonomik ilişkisini yansıtır. |
| **3** | `ortalama_model_yili` | %13,15 | Araçların ortalama model yılı. Araç yaşı, bakım ihtiyacı ve servis tercihiyle ilişkili olabilir. |
| **4** | `frequency` | %10,30 | Gözlem dönemindeki servis ziyaret sayısı. Müşterinin geçmiş servis alışkanlığını gösterir. |
| **5** | `sehir` | %7,09 | Müşterinin bulunduğu şehir. Servis ağına erişim ve bölgesel müşteri davranışlarıyla ilişkili olabilir. |

**Önemli not:** Bunlar Random Forest'ın ağaç yapılarından hesaplanan impurity-based feature importance değerleridir. Değişkenin tahminlere katkısını gösterir, nedensellik göstermez. Özellikle kategorik değişkenlerin önemini ve birbiriyle ilişkili özelliklerin paylaşılan önemini yorumlarken temkinli olmalıolunmalıdır.

In [9]:
from sklearn.inspection import permutation_importance

permutation_result = permutation_importance(
    rf_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

permutation_df = pd.DataFrame({
    "degisken": FEATURES,
    "importance_mean": permutation_result.importances_mean,
    "importance_std": permutation_result.importances_std
})

permutation_df = permutation_df.sort_values(
    by="importance_mean",
    ascending=False
).reset_index(drop=True)

print("PERMUTATION IMPORTANCE SIRALAMASI")
display(permutation_df.round(4))

print("\nİLK 5 DEĞİŞKEN")
display(permutation_df.head(5).round(4))

PERMUTATION IMPORTANCE SIRALAMASI


,degisken,importance_mean,importance_std
0,recency,0.0844,0.0163
1,ortalama_model_yili,0.0402,0.0105
2,musteri_tipi,0.0180,0.0088
3,frequency,0.0065,0.0064
4,monetary,0.0062,0.0059
5,kayit_kanali,0.0047,0.0058
6,aktif_garantili_arac_sayisi,0.0046,0.0050
7,sehir,0.0046,0.0074
8,arac_sayisi,0.0007,0.0014
9,farkli_kanal_sayisi,-0.0000,0.0010



İLK 5 DEĞİŞKEN


,degisken,importance_mean,importance_std
0,recency,0.0844,0.0163
1,ortalama_model_yili,0.0402,0.0105
2,musteri_tipi,0.0180,0.0088
3,frequency,0.0065,0.0064
4,monetary,0.0062,0.0059


**İki yöntemin karşılaştırılması**
| Değişken | Random Forest Önemi | Permutation Importance |
| --- | --- | --- |
| `recency` | %40,13 | 0,0844 |
| `monetary` | %14,96 | 0,0062 |
| `ortalama_model_yili` | %13,15 | 0,0402 |
| `frequency` | %10,30 | 0,0065 |
| `sehir` | %7,09 | 0,0046 |
| `musteri_tipi` | %4,23 | 0,0180 |

İki yöntemin ortak bulgusu:
- recency açık ara en güçlü değişken.
- ortalama_model_yili ikinci sırada.
- musteri_tipi, Permutation Importance yönteminde daha üst sıraya çıkıyor.
- monetary ve frequency, Random Forest'ın kendi önem ölçümünde daha yüksek görünürken, karıştırıldıklarında ROC-AUC üzerindeki etkileri daha sınırlı kalıyor.

In [10]:

rf_importance_save = grouped_importance[
    ["degisken", "importance", "importance_yuzde"]
].copy()

rf_importance_save = rf_importance_save.rename(columns={
    "importance": "rf_importance",
    "importance_yuzde": "rf_importance_yuzde"
})

permutation_importance_save = permutation_df.copy()

model_importance_report = rf_importance_save.merge(
    permutation_importance_save,
    on="degisken",
    how="outer"
)

model_importance_report = model_importance_report.sort_values(
    "importance_mean",
    ascending=False
).reset_index(drop=True)


importance_path = ANALIZ_DIR / "05_model_degisken_onemleri.csv"

model_importance_report.to_csv(
    importance_path,
    index=False,
    encoding="utf-8-sig"
)

print("Dosya kaydedildi:", importance_path)
print("Tablo boyutu:", model_importance_report.shape)
display(model_importance_report.round(4))

Dosya kaydedildi: c:\Users\ASUS\OneDrive\Masaüstü\borusan_automotive_case_study\ciktilar\analiz\05_model_degisken_onemleri.csv
Tablo boyutu: (12, 5)


,degisken,rf_importance,rf_importance_yuzde,importance_mean,importance_std
0,recency,0.4013,40.1312,0.0844,0.0163
1,ortalama_model_yili,0.1315,13.1465,0.0402,0.0105
2,musteri_tipi,0.0423,4.2281,0.0180,0.0088
3,frequency,0.1030,10.2994,0.0065,0.0064
4,monetary,0.1496,14.9629,0.0062,0.0059
5,kayit_kanali,0.0440,4.4022,0.0047,0.0058
6,aktif_garantili_arac_sayisi,0.0273,2.7316,0.0046,0.0050
7,sehir,0.0709,7.0861,0.0046,0.0074
8,arac_sayisi,0.0112,1.1158,0.0007,0.0014
9,farkli_kanal_sayisi,0.0063,0.6259,-0.0000,0.0010


In [11]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import Pipeline

# --------------------------------------------------
# 1. Denenecek modeller
# --------------------------------------------------

model_candidates = {
    "RF_Mevcut": rf_model,

    "RF_Daha_Sade": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "RF_Balanced_SubSample": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "GB_100": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=2,
            min_samples_leaf=10,
            random_state=42
        ))
    ]),

    "GB_200": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=10,
            random_state=42
        ))
    ])
}

# --------------------------------------------------
# 2. Aynı CV katları bütün modellerde kullanılacak
# --------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for model_name, model in model_candidates.items():

    print(f"Değerlendiriliyor: {model_name}")

    oof_probabilities = np.zeros(len(X_train))

    for train_idx, valid_idx in cv.split(X_train, y_train):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_valid = X_train.iloc[valid_idx]

        y_fold_train = y_train.iloc[train_idx]

        fold_model = clone(model)

        # Gradient Boosting için sınıf ağırlıkları
        if model_name.startswith("GB_"):
            sample_weights = compute_sample_weight(
                class_weight="balanced",
                y=y_fold_train
            )

            fold_model.fit(
                X_fold_train,
                y_fold_train,
                classifier__sample_weight=sample_weights
            )
        else:
            fold_model.fit(
                X_fold_train,
                y_fold_train
            )

        oof_probabilities[valid_idx] = (
            fold_model.predict_proba(X_fold_valid)[:, 1]
        )

    # Eğitim verisi üzerindeki out-of-fold tahminleri değerlendirilmesi
    predictions = (oof_probabilities >= 0.50).astype(int)

    results.append({
        "model": model_name,
        "roc_auc": roc_auc_score(y_train, oof_probabilities),
        "pr_auc": average_precision_score(y_train, oof_probabilities),
        "precision_050": precision_score(
            y_train, predictions, zero_division=0
        ),
        "recall_050": recall_score(
            y_train, predictions, zero_division=0
        ),
        "f1_050": f1_score(
            y_train, predictions, zero_division=0
        )
    })

# --------------------------------------------------
# 3. Sonuç tablosu
# --------------------------------------------------

model_comparison = pd.DataFrame(results)

model_comparison = model_comparison.sort_values(
    by="pr_auc",
    ascending=False
).reset_index(drop=True)

print("\nMODEL KARŞILAŞTIRMASI")
display(model_comparison.round(4))

# --------------------------------------------------
# 4. Sonuç kaydı
# --------------------------------------------------

comparison_path = ANALIZ_DIR / "05_model_karsilastirmasi.csv"

model_comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nKarşılaştırma dosyası kaydedildi:")
print(comparison_path)

Değerlendiriliyor: RF_Mevcut
Değerlendiriliyor: RF_Daha_Sade
Değerlendiriliyor: RF_Balanced_SubSample
Değerlendiriliyor: GB_100
Değerlendiriliyor: GB_200

MODEL KARŞILAŞTIRMASI


,model,roc_auc,pr_auc,precision_050,recall_050,f1_050
0,GB_200,0.7554,0.4980,0.3558,0.6217,0.4526
1,GB_100,0.7556,0.4976,0.3563,0.6198,0.4525
2,RF_Mevcut,0.7515,0.4601,0.3758,0.5837,0.4572
3,RF_Balanced_SubSample,0.7526,0.4597,0.4264,0.4905,0.4562
4,RF_Daha_Sade,0.7521,0.4539,0.3473,0.6160,0.4441



Karşılaştırma dosyası kaydedildi:
c:\Users\ASUS\OneDrive\Masaüstü\borusan_automotive_case_study\ciktilar\analiz\05_model_karsilastirmasi.csv


**Not:** Bu tablo eğitim verisindeki 5 katlı çapraz doğrulama tahminlerine aittir. Daha önceki 0,7144 ROC-AUC ve 0,4651 PR-AUC değerleri bağımsız test setindeki mevcut Random Forest sonuçlarıdır. Dolayısıyla bu iki grubu doğrudan aynı değerlendirme gibi karşılaştırmamalıyız.

| Rol | Model | Gerekçe / Avantaj |
| :--- | :--- | :--- |
| **Ana Aday** | `GB_100` | GB_200 ile performansı neredeyse aynı; daha az ağaç sayısı sayesinde hesaplama maliyetini düşürüyor. |
| **Alternatif** | `RF_Mevcut` | F1 skoru bazında bir miktar daha yüksek performans sunuyor. |

In [13]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)
import numpy as np
import pandas as pd


threshold_models = {
    "RF_Mevcut": rf_model,
    "GB_100": model_candidates["GB_100"]
}

cv_threshold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

threshold_results = []

for model_name, model in threshold_models.items():

    oof_probabilities = np.zeros(len(X_train))

    for train_idx, valid_idx in cv_threshold.split(X_train, y_train):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_valid = X_train.iloc[valid_idx]
        y_fold_train = y_train.iloc[train_idx]

        fold_model = clone(model)

        if model_name == "GB_100":
            sample_weights = compute_sample_weight(
                class_weight="balanced",
                y=y_fold_train
            )

            fold_model.fit(
                X_fold_train,
                y_fold_train,
                classifier__sample_weight=sample_weights
            )
        else:
            fold_model.fit(
                X_fold_train,
                y_fold_train
            )

        oof_probabilities[valid_idx] = (
            fold_model.predict_proba(X_fold_valid)[:, 1]
        )

    for threshold in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:

        predictions = (
            oof_probabilities >= threshold
        ).astype(int)

        threshold_results.append({
            "model": model_name,
            "threshold": threshold,
            "precision": precision_score(
                y_train, predictions, zero_division=0
            ),
            "recall": recall_score(
                y_train, predictions, zero_division=0
            ),
            "f1": f1_score(
                y_train, predictions, zero_division=0
            ),
            "flagged_customer_count": int(predictions.sum())
        })

threshold_analysis = pd.DataFrame(threshold_results)

print("MODEL VE EŞİK KARŞILAŞTIRMASI")
display(
    threshold_analysis
    .sort_values(["model", "threshold"])
    .round(4)
)

threshold_path = ANALIZ_DIR / "05_model_esik_karsilastirmasi.csv"

threshold_analysis.to_csv(
    threshold_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nDosya kaydedildi:", threshold_path)

MODEL VE EŞİK KARŞILAŞTIRMASI


,model,threshold,precision,recall,f1,flagged_customer_count
7,GB_100,0.35,0.2291,0.8498,0.3609,1951
8,GB_100,0.40,0.2659,0.7700,0.3953,1523
9,GB_100,0.45,0.3037,0.6882,0.4214,1192
10,GB_100,0.50,0.3563,0.6198,0.4525,915
11,GB_100,0.55,0.3922,0.5323,0.4516,714
12,GB_100,0.60,0.4652,0.4449,0.4548,503
13,GB_100,0.65,0.5209,0.3783,0.4383,382
0,RF_Mevcut,0.35,0.2461,0.7738,0.3734,1654
1,RF_Mevcut,0.40,0.2915,0.7205,0.4151,1300
2,RF_Mevcut,0.45,0.3404,0.6673,0.4509,1031



Dosya kaydedildi: c:\Users\ASUS\OneDrive\Masaüstü\borusan_automotive_case_study\ciktilar\analiz\05_model_esik_karsilastirmasi.csv


In [15]:
from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd

# 1. GB_100 modelini eğitim verisi ile eğitimi
gb_final = clone(model_candidates["GB_100"])

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

gb_final.fit(
    X_train,
    y_train,
    classifier__sample_weight=sample_weights
)

# 2. Test seti olasılık tahminleri
y_prob_gb = gb_final.predict_proba(X_test)[:, 1]

# 3. Eşik bağımsız performans ölçütleri
roc_auc = roc_auc_score(y_test, y_prob_gb)
pr_auc = average_precision_score(y_test, y_prob_gb)

print("GB_100 - TEST SETİ")
print("-" * 50)
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC : {pr_auc:.4f}")

# 4. Önceden belirlenen eşiklerde değerlendirme
test_results = []

for threshold in [0.35, 0.50, 0.60]:
    y_pred = (y_prob_gb >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    test_results.append({
        "model": "GB_100",
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "flagged_customer_count": int(y_pred.sum())
    })

gb_test_results = pd.DataFrame(test_results)

display(
    gb_test_results.round(4)
)

# 5. Kayıt
test_output_path = ANALIZ_DIR / "05_gb100_test_sonuclari.csv"

gb_test_results.to_csv(
    test_output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nDosya kaydedildi: {test_output_path}")

GB_100 - TEST SETİ
--------------------------------------------------
ROC-AUC: 0.7124
PR-AUC : 0.4623


,model,threshold,accuracy,precision,recall,f1,TN,FP,FN,TP,flagged_customer_count
0,GB_100,0.35,0.4906,0.2197,0.8106,0.3457,283,380,25,107,487
1,GB_100,0.50,0.7208,0.2955,0.4924,0.3693,508,155,67,65,220
2,GB_100,0.60,0.8164,0.4340,0.3485,0.3866,603,60,86,46,106



Dosya kaydedildi: c:\Users\ASUS\OneDrive\Masaüstü\borusan_automotive_case_study\ciktilar\analiz\05_gb100_test_sonuclari.csv


In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd

comparison_results = []

models_probabilities = {
    "RF_Mevcut": y_prob_rf,
    "GB_100": y_prob_gb
}

thresholds_to_compare = [0.35, 0.50, 0.55, 0.60]

for model_name, probabilities in models_probabilities.items():

    for threshold in thresholds_to_compare:

        predictions = (probabilities >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_test,
            predictions,
            labels=[0, 1]
        ).ravel()

        comparison_results.append({
            "model": model_name,
            "threshold": threshold,
            "accuracy": accuracy_score(y_test, predictions),
            "precision": precision_score(y_test, predictions, zero_division=0),
            "recall": recall_score(y_test, predictions, zero_division=0),
            "f1": f1_score(y_test, predictions, zero_division=0),
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "flagged_customer_count": int(predictions.sum())
        })

model_test_comparison = pd.DataFrame(comparison_results)

display(
    model_test_comparison.round(4)
)

comparison_path = ANALIZ_DIR / "05_rf_gb_test_karsilastirmasi.csv"

model_test_comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Dosya kaydedildi: {comparison_path}")

,model,threshold,accuracy,precision,recall,f1,TN,FP,FN,TP,flagged_customer_count
0,RF_Mevcut,0.35,0.5421,0.2327,0.7652,0.3569,330,333,31,101,434
1,RF_Mevcut,0.50,0.7560,0.3418,0.5076,0.4085,534,129,65,67,196
2,RF_Mevcut,0.55,0.7862,0.3812,0.4621,0.4178,564,99,71,61,160
3,RF_Mevcut,0.60,0.8214,0.4554,0.3864,0.4180,602,61,81,51,112
4,GB_100,0.35,0.4906,0.2197,0.8106,0.3457,283,380,25,107,487
5,GB_100,0.50,0.7208,0.2955,0.4924,0.3693,508,155,67,65,220
6,GB_100,0.55,0.7623,0.3273,0.4091,0.3636,552,111,78,54,165
7,GB_100,0.60,0.8164,0.4340,0.3485,0.3866,603,60,86,46,106


Dosya kaydedildi: c:\Users\ASUS\OneDrive\Masaüstü\borusan_automotive_case_study\ciktilar\analiz\05_rf_gb_test_karsilastirmasi.csv


In [17]:
importance_path = ANALIZ_DIR / "05_model_degisken_onemleri.csv"

print("Dosya mevcut mu:", importance_path.exists())

if importance_path.exists():
    importance_df = pd.read_csv(importance_path)

    display(
        importance_df.sort_values(
            "importance_mean",
            ascending=False
        ).round(4)
    )
else:
    print("Değişken önemleri dosyası bulunamadı.")

Dosya mevcut mu: True


,degisken,rf_importance,rf_importance_yuzde,importance_mean,importance_std
0,recency,0.4013,40.1312,0.0844,0.0163
1,ortalama_model_yili,0.1315,13.1465,0.0402,0.0105
2,musteri_tipi,0.0423,4.2281,0.0180,0.0088
3,frequency,0.1030,10.2994,0.0065,0.0064
4,monetary,0.1496,14.9629,0.0062,0.0059
5,kayit_kanali,0.0440,4.4022,0.0047,0.0058
6,aktif_garantili_arac_sayisi,0.0273,2.7316,0.0046,0.0050
7,sehir,0.0709,7.0861,0.0046,0.0074
8,arac_sayisi,0.0112,1.1158,0.0007,0.0014
9,farkli_kanal_sayisi,0.0063,0.6259,-0.0000,0.0010


* **Nihai Model Adayı:** Random Forest
* **Operasyonel Eşik Adayı:** 0.55
* **Model Performans Metrikleri:**
  * **Test ROC-AUC:** 0.7144
  * **Test PR-AUC:** 0.4651
  * **Test F1:** 0.4178
* **Önemli Değişkenler:**
  * **1. En Önemli:** `recency`
  * **2. En Önemli:** `ortalama_model_yili`

In [19]:
from pathlib import Path

MODEL_RAPOR_DOSYASI = ANALIZ_DIR / "05_churn_modelleme_raporu.md"

model_raporu = r"""
# 5. Churn Tahmin Modeli

## 5.1. Problem tanımı ve hedef değişken

Churn tahmini, 30 Haziran 2025 itibarıyla geçmişte en az bir servis kaydı bulunan müşteriler üzerinden gerçekleştirilmiştir.

- **Özellik gözlem dönemi:** 01.01.2022–30.06.2025
- **Özellik kesim tarihi:** 30.06.2025
- **Hedef dönem:** 01.07.2025–30.06.2026
- **Churn = 1:** Hedef dönemde servis kaydı bulunmayan müşteri.
- **Churn = 0:** Hedef dönemde en az bir servis kaydı bulunan müşteri.

Analiz popülasyonu 3.974 müşteriden oluşmaktadır. Bunların 3.316'sı (%83,44) churn değil, 658'i (%16,56) churn olarak etiketlenmiştir.

Hedef dönemdeki servis kayıtları yalnızca hedef değişkeni oluşturmak için kullanılmış, model özelliklerine dahil edilmemiştir.

## 5.2. Özellik seçimi ve veri sızıntısı kontrolleri

Modelde 12 özellik kullanılmıştır:

- Kategorik: `sehir`, `musteri_tipi`, `kayit_kanali`
- Sayısal: `frequency`, `monetary`, `recency`, `arac_sayisi`, `ortalama_model_yili`, `aktif_garantili_arac_sayisi`, `geri_bildirim_sayisi`, `metni_olan_geri_bildirim_sayisi`, `farkli_kanal_sayisi`

Müşteri kimliği, son servis tarihi, geri bildirim tarihleri ve hedef dönem servis sayısı özelliklerden çıkarılmıştır. Özellikle hedef dönem servis sayısının modele dahil edilmesi doğrudan hedef sızıntısına yol açabileceğinden engellenmiştir.

Recency değerleri ile son servis tarihi arasındaki tutarlılık kontrol edilmiş; geçersiz tarih veya gün farkı uyumsuzluğu bulunmamıştır. Seçilen özelliklerde eksik değer gözlenmemiştir.

Kategorik değişkenler One-Hot Encoding ile dönüştürülmüş, sayısal değişkenler mevcut değerleriyle modele aktarılmıştır.

## 5.3. Eğitim ve test ayrımı

Veri, churn sınıf oranları korunarak stratified yöntemle %80 eğitim ve %20 test olarak ayrılmıştır. Ayrım `random_state=42` ile tekrarlanabilir hale getirilmiştir.

- Eğitim: 3.179 müşteri
- Test: 795 müşteri

Model karşılaştırması eğitim verisi üzerinde 5 katlı Stratified K-Fold çapraz doğrulama ile yapılmıştır. Test seti nihai değerlendirme için ayrılmıştır.

Bu veri setinde tek bir hedef dönemi bulunduğundan farklı tarihsel dönemlerde tekrarlanan zaman bazlı doğrulama yapılamamıştır. Bu nedenle rastgele bölme sonuçları, gelecekteki dönemlerdeki performansın garantisi değildir.

## 5.4. Baseline modeli

En sık sınıfı tahmin eden DummyClassifier baseline modeli test setinde %83,40 doğruluk elde etmiştir. Ancak churn sınıfını hiç tahmin etmemiştir:

- Churn precision: 0
- Churn recall: 0
- Churn F1: 0

Sınıf dağılımının dengesiz olması nedeniyle yalnızca accuracy değerine bakmanın yanıltıcı olabileceği görülmüştür. Bu nedenle model değerlendirmesinde precision, recall, F1 ve PR-AUC metrikleri de kullanılmıştır.

## 5.5. Model karşılaştırması

Random Forest ve Gradient Boosting modelleri aynı özellikler ve aynı 5 katlı çapraz doğrulama düzeniyle karşılaştırılmıştır. Gradient Boosting modellerinde sınıf dengesizliğini ele almak için her eğitim katında dengeli örnek ağırlıkları kullanılmıştır.

Eğitim çapraz doğrulama sonuçları:

| Model | ROC-AUC | PR-AUC | F1 (0,50) |
|---|---:|---:|---:|
| Random Forest - mevcut | 0,7515 | 0,4601 | 0,4572 |
| Gradient Boosting - 100 | 0,7556 | 0,4976 | 0,4525 |
| Gradient Boosting - 200 | 0,7554 | 0,4980 | 0,4526 |

Gradient Boosting modelleri PR-AUC açısından daha yüksek sonuç vermiştir. Random Forest ise 0,50 eşikte az farkla daha yüksek F1 elde etmiştir. Modellerin sonuçları birbirine yakın olduğundan yalnızca tek bir çapraz doğrulama metriğine göre karar verilmemiştir.

Aynı bağımsız test setindeki değerlendirmede Random Forest, Gradient Boosting modeline göre ROC-AUC, PR-AUC ve karşılaştırılan eşiklerde F1 açısından az farkla daha iyi sonuç vermiştir. Bu nedenle nihai model adayı olarak Random Forest seçilmiştir.

## 5.6. Nihai modelin test performansı

Random Forest test sonuçları:

- ROC-AUC: 0,7144
- PR-AUC: 0,4651

Eşik karşılaştırması:

| Eşik | Accuracy | Precision | Recall | F1 | TN | FP | FN | TP | İşaretlenen müşteri |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 0,50 | 0,7560 | 0,3418 | 0,5076 | 0,4085 | 534 | 129 | 65 | 67 | 196 |
| 0,55 | 0,7862 | 0,3812 | 0,4621 | 0,4178 | 564 | 99 | 71 | 61 | 160 |
| 0,60 | 0,8214 | 0,4554 | 0,3864 | 0,4180 | 602 | 61 | 81 | 51 | 112 |

Eşik seçimi için eğitim verisindeki çapraz doğrulama sonuçları esas alınmıştır. 0,50 eşiği en yüksek F1 değerini (0,4572) vermiştir. 0,55 eşiğinin F1 değeri (0,4567) buna çok yakındır; ancak daha az müşteriyi işaretlemektedir.

Bu nedenle 0,55 eşiği, müşteri iletişim kapasitesi sınırlı olabilecek bir operasyon için **geçici operasyonel aday** olarak değerlendirilmiştir. 0,50 eşiği ise daha yüksek recall hedeflenen alternatif olarak korunmuştur. Bu karar, gerçek iletişim kapasitesi ve yanlış pozitif/yanlış negatif maliyetleri bilinmeden kesin optimum olarak yorumlanmamalıdır.

0,55 eşikte test setindeki 160 işaretli müşterinin 61'i gerçekten churn olmuş, 99'u churn olmamıştır. Ayrıca 71 churn müşterisi kaçırılmıştır. Bu sonuçlar modelin tek başına otomatik kampanya veya müşteri kararı vermesi için yeterli olmadığını göstermektedir.

## 5.7. Değişken önemleri

Değişken önemleri iki yöntemle incelenmiştir. Yorumlamada test seti üzerindeki permutation importance değerleri önceliklendirilmiştir.

| Sıra | Değişken | Permutation importance |
|---:|---|---:|
| 1 | recency | 0,0844 |
| 2 | ortalama_model_yili | 0,0402 |
| 3 | musteri_tipi | 0,0180 |
| 4 | frequency | 0,0065 |
| 5 | monetary | 0,0062 |

Recency en yüksek katkıyı sağlamıştır. Son servisinden itibaren geçen sürenin artması, churn tahmininde güçlü bir sinyal olarak öne çıkmaktadır. Ortalama araç model yılı ve müşteri tipi de katkı sağlamıştır.

Diğer değişkenlerin permutation importance değerleri görece düşüktür. Sıfıra yakın veya negatif değerler, test setinde ölçülebilir katkı görülmediğini gösterebilir; bu değişkenlerin churn üzerinde olumsuz veya nedensel etkisi olduğu anlamına gelmez.

Değişken önemleri modelin tahmin davranışını açıklar; nedensellik göstermez.

## 5.8. Sınırlamalar ve iş açısından değerlendirme

1. **Zaman bazlı doğrulama:** Tek hedef dönemi bulunduğundan farklı dönemlerde model kararlılığı test edilememiştir.
2. **Sınıf dengesizliği:** Churn oranı %16,56'dır. Accuracy tek başına yeterli değerlendirme ölçütü değildir.
3. **Yanlış pozitifler:** 0,55 eşikte 99 churn olmayan müşteri riskli işaretlenmiştir. Bu müşterilere yapılacak iletişimin maliyeti ve müşteri deneyimine etkisi değerlendirilmelidir.
4. **Yanlış negatifler:** Aynı eşikte 71 churn müşterisi kaçırılmıştır. Yüksek riskli müşterilerin kaçırılmasının iş maliyeti ayrıca ele alınmalıdır.
5. **Genellenebilirlik:** Test ROC-AUC değeri 0,7144'tür. Modelin ayırt etme gücü sınırlıdır ve üretim ortamına alınmadan önce yeni dönem verileriyle doğrulanmalıdır.
6. **Nedensellik:** Model ilişkileri ve tahmin sinyallerini öğrenir; belirli bir müşteri davranışının churn'e neden olduğunu kanıtlamaz.

İş açısından model, müşteri temsilcilerine inceleme önceliği oluşturabilecek bir karar destek aracı olarak değerlendirilebilir. Müşteriye yönelik aksiyonlar, model tahmininin yanı sıra müşteri geçmişi ve geri bildirimleri insan kontrolüyle incelenerek belirlenmelidir.

Üretim öncesinde zaman bazlı ileri dönem testi, eşik ve maliyet analizi, model izleme ve periyodik performans kontrolü yapılması önerilir.
"""

MODEL_RAPOR_DOSYASI.parent.mkdir(parents=True, exist_ok=True)
MODEL_RAPOR_DOSYASI.write_text(model_raporu.strip(), encoding="utf-8")

print("Rapor oluşturuldu:", MODEL_RAPOR_DOSYASI)
print("Dosya mevcut mu:", MODEL_RAPOR_DOSYASI.exists())
print("Dosya boyutu (KB):", round(MODEL_RAPOR_DOSYASI.stat().st_size / 1024, 2))

Rapor oluşturuldu: c:\Users\ASUS\OneDrive\Masaüstü\borusan_automotive_case_study\ciktilar\analiz\05_churn_modelleme_raporu.md
Dosya mevcut mu: True
Dosya boyutu (KB): 7.83


In [23]:
print("X_test index örneği:")
print(X_test.index[:10].tolist())

print("\nX_test index aralığı:")
print(X_test.index.min(), "-", X_test.index.max())

print("\nNotebook'taki müşteri tablosu olabilecek DataFrame'ler:")

for degisken, deger in globals().copy().items():
    if isinstance(deger, pd.DataFrame):
        if "musteri_id" in deger.columns:
            print(
                f"{degisken}: shape={deger.shape}, "
                f"index_unique={deger.index.is_unique}, "
                f"musteri_id_unique={deger['musteri_id'].is_unique}"
            )

X_test index örneği:
[1723, 2216, 1158, 2155, 3108, 1791, 500, 1291, 3964, 3528]

X_test index aralığı:
1 - 3964

Notebook'taki müşteri tablosu olabilecek DataFrame'ler:
musteri_df: shape=(3974, 18), index_unique=True, musteri_id_unique=True


In [24]:
import pandas as pd
import numpy as np


assert len(X_test) == len(y_test) == len(y_prob_test), \
    "Test verisi ve tahmin uzunlukları eşleşmiyor."

# Test indekslerinin müşteri tablosunda bulunduğunu doğrulama
assert X_test.index.isin(musteri_df.index).all(), \
    "Bazı test indeksleri müşteri tablosunda bulunamadı."

#  Müşteri bazlı tahmin tablosu
tahmin_sonuclari = pd.DataFrame({
    "musteri_id": musteri_df.loc[X_test.index, "musteri_id"].values,
    "gercek_churn": np.asarray(y_test),
    "churn_olasiligi": np.asarray(y_prob_test)
}, index=X_test.index)


tahmin_sonuclari["tahmin_churn"] = (
    tahmin_sonuclari["churn_olasiligi"] >= 0.55
).astype(int)


tahmin_sonuclari = tahmin_sonuclari.sort_values(
    "churn_olasiligi", ascending=False
).reset_index(drop=True)


TAHMIN_DOSYASI = ANALIZ_DIR / "05_musteri_churn_tahminleri.csv"

tahmin_sonuclari.to_csv(
    TAHMIN_DOSYASI,
    index=False,
    encoding="utf-8-sig"
)


print("Dosya kaydedildi:", TAHMIN_DOSYASI)
print("Toplam müşteri:", len(tahmin_sonuclari))
print("Benzersiz müşteri:", tahmin_sonuclari["musteri_id"].nunique())
print("\nGerçek churn dağılımı:")
print(tahmin_sonuclari["gercek_churn"].value_counts().sort_index())

print("\nTahmin edilen yüksek riskli müşteri sayısı (eşik 0.55):",
      tahmin_sonuclari["tahmin_churn"].sum())

print("\nEn yüksek riskli ilk 10 müşteri:")
display(tahmin_sonuclari.head(10))

Dosya kaydedildi: c:\Users\ASUS\OneDrive\Masaüstü\borusan_automotive_case_study\ciktilar\analiz\05_musteri_churn_tahminleri.csv
Toplam müşteri: 795
Benzersiz müşteri: 795

Gerçek churn dağılımı:
gercek_churn
0    663
1    132
Name: count, dtype: int64

Tahmin edilen yüksek riskli müşteri sayısı (eşik 0.55): 160

En yüksek riskli ilk 10 müşteri:


,musteri_id,gercek_churn,churn_olasiligi,tahmin_churn
0,100687,1,0.915367,1
1,100036,1,0.911669,1
2,102219,1,0.909814,1
3,101580,1,0.904683,1
4,102087,1,0.903756,1
5,103214,1,0.897111,1
6,101530,1,0.892185,1
7,101218,1,0.888071,1
8,101074,1,0.886238,1
9,104020,1,0.865303,1
